In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.tseries.offsets import DateOffset
from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')

FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future
LAGS = [1, 2, 7, DateOffset(months=1), DateOffset(months=3), DateOffset(months=6)]
DIFFS = [1, DateOffset(months=1), DateOffset(months=3), DateOffset(months=6)]
ROLL_WINDOWS = { 7: 1,                                                              # window: lags
                30: [DateOffset(months=1), DateOffset(months=3), DateOffset(months=6)]}


# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always


In [10]:
store_df = pd.read_csv(STORE_FILE)
store_df = process_store_data(store_df)

df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
df_train_store = attach_store_data(df_train, store_df)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store, lags=LAGS, roll_windows=ROLL_WINDOWS, diffs=DIFFS)
targets = make_targets(df=df_train[['Date', 'Store', 'Sales']], horizon=FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_22164\3323829162.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [11]:
pd.concat([
    df_features.dtypes,
    df_features.isna().sum()/len(df_features),
    df_features.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('feature_summary.csv')

In [12]:
df_features.shape, targets.shape

((1017209, 64), (1017209, 44))